# Download OSM street network (Cook County)

r5py routes transit trips over an **OSM street network** for the walk access, egress and
transfer legs — GTFS alone is not enough. This notebook produces that input once and saves it
to `data/cook_county.osm.pbf`, which `Make Skims.ipynb` then reads.

There is no county-level OSM download for the US, so the steps are:

1. Download the statewide **Illinois** extract from [Geofabrik](https://download.geofabrik.de/north-america/us/illinois.html).
2. **Clip** it to a Cook County bounding box with `osmium`.

Run this from the `rh_pricing` conda environment, which must include `osmium-tool`:

```
conda install -n rh_pricing -c conda-forge osmium-tool
```

In [1]:
import os, subprocess, urllib.request

DATA_DIR      = "data"
ILLINOIS_PBF  = os.path.join(DATA_DIR, "illinois-latest.osm.pbf")
COOK_PBF      = os.path.join(DATA_DIR, "cook_county.osm.pbf")
GEOFABRIK_URL = "https://download.geofabrik.de/north-america/us/illinois-latest.osm.pbf"

# Cook County bounding box, lightly padded so walk legs near the border aren't clipped.
# osmium -b expects: left,bottom,right,top  ==  West,South,East,North
BBOX = {"W": -88.30, "S": 41.45, "E": -87.50, "N": 42.16}

os.makedirs(DATA_DIR, exist_ok=True)
print("Output target:", COOK_PBF)

Output target: data\cook_county.osm.pbf


## 1. Download the Illinois extract from Geofabrik

~250–400 MB. Skipped automatically if the file already exists.

In [2]:
if os.path.exists(ILLINOIS_PBF):
    print(f"Already present: {ILLINOIS_PBF} ({os.path.getsize(ILLINOIS_PBF)/1e6:.0f} MB)")
else:
    print(f"Downloading {GEOFABRIK_URL}")

    def _progress(blocks, block_size, total):
        done = blocks * block_size
        pct = 100 * done / total if total > 0 else 0
        print(f"\r  {done/1e6:7.0f} / {total/1e6:7.0f} MB ({pct:5.1f}%)", end="")

    urllib.request.urlretrieve(GEOFABRIK_URL, ILLINOIS_PBF, reporthook=_progress)
    print(f"\nSaved {ILLINOIS_PBF} ({os.path.getsize(ILLINOIS_PBF)/1e6:.0f} MB)")

      351 /     351 MB (100.0%)
Saved data\illinois-latest.osm.pbf (351 MB)


## 2. Clip to Cook County with osmium

A bounding-box clip is fast and keeps a little area beyond the county line, which is
helpful for walk-egress near the border. The result is the only OSM input the skim
notebook needs.

In [3]:
bbox_str = f"{BBOX['W']},{BBOX['S']},{BBOX['E']},{BBOX['N']}"
cmd = ["osmium", "extract", "-b", bbox_str, ILLINOIS_PBF, "-o", COOK_PBF, "--overwrite"]
print("Running:", " ".join(cmd))

try:
    subprocess.run(cmd, check=True)
except FileNotFoundError:
    raise SystemExit(
        "osmium not found. Install it into this environment:\n"
        "    conda install -n rh_pricing -c conda-forge osmium-tool"
    )

print(f"\nSaved {COOK_PBF} ({os.path.getsize(COOK_PBF)/1e6:.1f} MB)")

Running: osmium extract -b -88.3,41.45,-87.5,42.16 data\illinois-latest.osm.pbf -o data\cook_county.osm.pbf --overwrite

Saved data\cook_county.osm.pbf (144.2 MB)


## 3. (Optional) Reclaim disk space

The full Illinois file is only needed to produce the clip. Uncomment to delete it.

In [4]:
# os.remove(ILLINOIS_PBF)
# print("Removed", ILLINOIS_PBF)